# Anatomy of a Redemption
### Spark-Based Multi-Source Sentiment Analysis of Cyberpunk 2077

Lab Group 1 — Data Processing 2, SS 2026

Batch pipeline: Reddit comments + Steam reviews over the 34-month window
(Dec 2020 – Dec 2023), sentiment via VADER, aligned against a patch timeline.
**Core thesis:** Reddit recovers faster than Steam.

**Stages:** 0 Config · 1 Ingest+Filter · 2 Cleaning · 3 VADER · 4 Aggregation · 5 Analysis/Plots

> Final, aggregated results go to `results/` (small, in the repo). The report is written from
> these files, not by re-running the pipeline.

## 0 · Configuration & Parameters (central)

All settings in **one block** — prerequisite for reproducibility. `DATA_MODE`
is the only switch between sample (development) and full run (`results/` snapshot). `SEED`
covers everything stochastic (especially LDA).

In [1]:
import findspark
findspark.init()

from pathlib import Path
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# ===== Central parameters (one block — reproducibility) =====
# NOTE: "full" for now because no sample has been carved yet — the real data
# (currently only r/cyberpunkgame) lives under data/raw/. Switch to "sample"
# once data/sample/{reddit,steam}/ is populated for fast iteration.
DATA_MODE = "full"             # "sample" | "full"
SEED = 42                       # for everything stochastic (esp. LDA setSeed)
assert DATA_MODE in ("sample", "full")

WINDOW_START = "2020-12-01"     # 34-month window, inclusive bounds
WINDOW_END   = "2023-12-31"
SUBREDDITS   = ["cyberpunkgame", "LowSodiumCyberpunk", "Games", "pcgaming"]
STEAM_APP_ID = 1091500          # Cyberpunk 2077

ANOMALY_Z_THRESHOLD = 3.0       # volume anomaly / review-bombing flag
LDA_NUM_TOPICS = 8              # topic analysis

# ===== Resolve project root robustly (cross-platform: macOS + Windows) =====
# Notebooks have no __file__; the Jupyter CWD is sometimes the project root and
# sometimes notebooks/. So we walk up from the CWD to find the project root.
def _find_project_root(start: Path) -> Path:
    for p in (start, *start.parents):
        if (p / "notebooks").is_dir() and (p / "README.md").exists():
            return p
    return start

PROJECT_ROOT = _find_project_root(Path.cwd().resolve())

# ===== Paths (POSIX strings -> accepted by Spark on macOS AND Windows) =====
_SUB           = "sample" if DATA_MODE == "sample" else "raw"
RAW_REDDIT     = (PROJECT_ROOT / "data" / _SUB / "reddit").as_posix() + "/"
RAW_STEAM      = (PROJECT_ROOT / "data" / _SUB / "steam").as_posix() + "/"
PROCESSED      = (PROJECT_ROOT / "data" / "processed").as_posix() + "/"
RESULTS        = (PROJECT_ROOT / "results").as_posix() + "/"
FIGURES        = (PROJECT_ROOT / "results" / "figures").as_posix() + "/"
# Patch timeline: small + hand-maintained -> lives in the repo (data/sample/), for sample AND full
PATCH_TIMELINE = (PROJECT_ROOT / "data" / "sample" / "patch_timeline.csv").as_posix()
Path(RESULTS).mkdir(exist_ok=True)
Path(FIGURES).mkdir(parents=True, exist_ok=True)

print(f"PROJECT_ROOT = {PROJECT_ROOT}")
print(f"DATA_MODE={DATA_MODE}  window={WINDOW_START}..{WINDOW_END}  seed={SEED}")

PROJECT_ROOT = /Users/luwu/Github/Projects/Data Processing 2 
DATA_MODE=full  window=2020-12-01..2023-12-31  seed=42


### SparkSession
Local, `local[*]`. Driver RAM increased for the `full` run.

In [2]:
import os, sys

# Help the kernel find Java + the matching Python (no-op if already set).
# Spark needs JAVA_HOME; on macOS Homebrew installs it in a predictable place.
if not os.environ.get("JAVA_HOME"):
    for _jh in ("/opt/homebrew/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home",
                "/usr/local/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home"):
        if Path(_jh).exists():
            os.environ["JAVA_HOME"] = _jh
            break
os.environ.setdefault("PYSPARK_PYTHON", sys.executable)

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("CyberpunkSentiment")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.shuffle.partitions", "200")          # increase for full run if needed
    .config("spark.driver.memory", "8g" if DATA_MODE == "sample" else "12g")
    .config("spark.sql.files.maxPartitionBytes", "128m")    # controls partition size
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/06/11 02:44:28 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## 1 · Ingest + Immediate Filter

**Hard rule:** filter immediately on read (subreddits + time window), before anything else.
GB/month should become a fraction early.

Three inputs, unified to a common schema `source, month, date, text`:
- **Reddit comments** (`body`) + **submissions** (`title` + `selftext`) — both included.
- **Steam reviews** — verified schema (`Review`, `Date Posted`).
- Patch timeline (below) is a small side input, not part of the document corpus.

> Currently only **r/cyberpunkgame** is downloaded; the design already scales to all 4 subreddits
> (just drop their dumps into the same folder).

### 1a · Decompress Reddit dumps (`.zst` → `.jsonl`)

The pip PySpark build ships Hadoop **without the native zstd codec**, so Spark cannot read `.zst`
directly. We decompress once to `.jsonl` (cross-platform via the `zstandard` package; cached —
skipped if already present). The `.jsonl` are written next to the dumps under the same folder
(git-ignored). Note: the decompressed files are large (~15 GB for the comments dump).

In [3]:
import zstandard

def ensure_jsonl(zst_path: Path) -> Path:
    """Decompress a .zst dump to .jsonl once (idempotent; skipped if present)."""
    out = zst_path.with_suffix(".jsonl")
    if out.exists() and out.stat().st_size > 0:
        return out
    print("decompressing", zst_path.name, "...")
    dctx = zstandard.ZstdDecompressor(max_window_size=2**31)   # dumps use a long window
    with open(zst_path, "rb") as fin, open(out, "wb") as fout:
        dctx.copy_stream(fin, fout, read_size=2**24, write_size=2**24)
    return out

for _zst in sorted(Path(RAW_REDDIT).glob("*.zst")):
    print("ready:", ensure_jsonl(_zst).name)

ready: cyberpunkgame_comments.jsonl
ready: cyberpunkgame_submissions.jsonl


In [4]:
# -------------------------------------------------------------
# Reddit: comments (body) + submissions (title + selftext), unified.
# Reads the decompressed *.jsonl from the previous cell. Explicit schema
# avoids slow inference over GBs. created_utc may be "123" or "123.0"
# (float string) -> cast double then long.
# -------------------------------------------------------------
from pyspark.sql import types as T

START = F.unix_timestamp(F.lit(WINDOW_START), "yyyy-MM-dd")
END   = F.unix_timestamp(F.lit(WINDOW_END),   "yyyy-MM-dd") + F.lit(86400)   # include the last day

_comment_schema = T.StructType([
    T.StructField("subreddit",   T.StringType()),
    T.StructField("body",        T.StringType()),
    T.StructField("created_utc", T.StringType()),
])
_submission_schema = T.StructType([
    T.StructField("subreddit",   T.StringType()),
    T.StructField("title",       T.StringType()),
    T.StructField("selftext",    T.StringType()),
    T.StructField("created_utc", T.StringType()),
])

def _in_window(df):
    df = df.withColumn("created_utc", F.col("created_utc").cast("double").cast("long"))
    return df.filter(
        F.col("subreddit").isin(SUBREDDITS)            # safety net; design scales to all 4 subreddits
        & (F.col("created_utc") >= START) & (F.col("created_utc") < END)
    )

comments = (
    _in_window(spark.read.schema(_comment_schema).option("pathGlobFilter", "*_comments.jsonl").json(RAW_REDDIT))
    .select("created_utc", F.col("body").alias("text"))
)
submissions = (
    _in_window(spark.read.schema(_submission_schema).option("pathGlobFilter", "*_submissions.jsonl").json(RAW_REDDIT))
    .select("created_utc", F.concat_ws(" ", F.col("title"), F.col("selftext")).alias("text"))
)

reddit = (
    comments.unionByName(submissions)
    .filter(F.col("text").isNotNull() & ~F.trim("text").isin("", "[deleted]", "[removed]"))
    .withColumn("date",  F.to_date(F.from_unixtime("created_utc")))
    .withColumn("month", F.date_format("date", "yyyy-MM"))
    .withColumn("source", F.lit("reddit"))
    .select("source", "month", "date", "text")
)
reddit.show(3, truncate=80)   # cheap peek; full counts come after the write below

+------+-------+----------+-----------------------------------+
|source|  month|      date|                               text|
+------+-------+----------+-----------------------------------+
|reddit|2020-12|2020-12-01|              First world problems.|
|reddit|2020-12|2020-12-01|This post did. Mortally wounded me.|
|reddit|2020-12|2020-12-01|   *Shortages of digital editions?*|
+------+-------+----------+-----------------------------------+
only showing top 3 rows


In [5]:
# -------------------------------------------------------------
# Steam reviews (CSV). Verified schema:
#   ReviewID, SteamID, Rating, Review, "Date Posted", Hours Played, ...
# multiLine + escape='"' because reviews contain newlines and doubled quotes.
# "Date Posted" has TWO formats in this dataset: mostly M/d/yyyy (e.g. 12/9/2020),
# a minority d-MMM-yy (e.g. 8-Aug-24). try_to_timestamp returns NULL on mismatch
# (Spark 4 is strict/ANSI), so we coalesce both patterns; anything else -> NULL -> dropped.
# (Rating = Recommended/Not Recommended kept aside for later VADER validation.)
# -------------------------------------------------------------
_date_expr = (
    "coalesce("
    " to_date(try_to_timestamp(`Date Posted`, 'M/d/yyyy')),"
    " to_date(try_to_timestamp(`Date Posted`, 'd-MMM-yy'))"
    ")"
)
steam = (
    spark.read
    .option("header", True).option("multiLine", True).option("escape", '"')
    .csv(RAW_STEAM)
    .select(F.col("Review").alias("text"), F.expr(_date_expr).alias("date"))
    .filter(F.col("text").isNotNull() & (F.length(F.trim("text")) > 0))
    .filter(F.col("date").between(WINDOW_START, WINDOW_END))   # drops out-of-window (e.g. 2024) + unparsable dates
    .withColumn("month", F.date_format("date", "yyyy-MM"))
    .withColumn("source", F.lit("steam"))
    .select("source", "month", "date", "text")
)
steam.show(3, truncate=80)   # cheap peek; full counts come after the write below

+------+-------+----------+---------------------------------------------------------------------------------------------------------------------------------+
|source|  month|      date|                                                                                                                             text|
+------+-------+----------+---------------------------------------------------------------------------------------------------------------------------------+
| steam|2020-12|2020-12-09|                                                 \n\t\t\t\t\t"Hᴇ's ғᴜᴄᴋᴇᴅ ɪɴ ᴛʜᴇ ʜᴇᴀᴅ, ᴛʜᴇ ᴡᴏʀʟᴅ's ғᴜᴄᴋᴇᴅ ɪɴ ᴛʜᴇ ʜᴇᴀᴅ - ᴀɴᴅ ʏᴏ...|
| steam|2020-12|2020-12-09|                                                  \n\t\t\t\t\tIt's a city of dreams... and I'm a big dreamer.\t\t\t\t\t\n\t\t\t\t|
| steam|2020-12|2020-12-09|\n\t\t\t\t\t======赛博朋克2077======千呼万唤始出来，赛博朋克终问世；夜城诱惑惹人厌，风流轶事留民间；义肢持枪逞英雄，惩恶扬善辨...|
+------+-------+----------+--------------------------------------------------------------------------

In [6]:
# -------------------------------------------------------------
# Combine both sources and persist (partitioned by source + month).
# The write is the one heavy pass; read the small parquet back for stats
# so we don't re-scan the raw data just to count.
# Continue downstream with: spark.read.parquet(PROCESSED + "cleaned/")
# NEVER .collect()/.toPandas() on the full df — aggregate first.
# -------------------------------------------------------------
documents = reddit.unionByName(steam)
(
    documents.write.mode("overwrite")
    .partitionBy("source", "month")
    .parquet(PROCESSED + "cleaned/")
)
print("Written to:", PROCESSED + "cleaned/")

cleaned = spark.read.parquet(PROCESSED + "cleaned/")
cleaned.groupBy("source").count().orderBy("source").show()
print("Total documents:", cleaned.count())

Written to: /Users/luwu/Github/Projects/Data Processing 2 /data/processed/cleaned/


+------+-------+
|source|  count|
+------+-------+
|reddit|5616435|
| steam| 691540|
+------+-------+

Total documents: 6307975


### 1c · Patch Timeline (3rd Data Source)

Small, manually maintained table (`patch_name, release_date, platform, notes`) with the major
Cyberpunk 2077 patches in the window (1.0 Launch … 2.1). Stored in the repo under `data/sample/`.
Used later as patch markers on the sentiment time series and as anchor for the recovery metric.

In [7]:
# -------------------------------------------------------------
# Read Patch Timeline (3rd data source)
#   Small + manually maintained -> in the repo. Used later for
#   patch markers and the recovery metric (core thesis).
# -------------------------------------------------------------
patches = (
    spark.read.option("header", True).csv(PATCH_TIMELINE)
    .withColumn("release_date", F.to_date("release_date"))
    .withColumn("month", F.date_format("release_date", "yyyy-MM"))
    .filter(F.col("release_date").between(WINDOW_START, WINDOW_END))
)
patches.orderBy("release_date").show(truncate=False)

+-----------------+------------+-------------+------------------------------------------------------------------------------+-------+
|patch_name       |release_date|platform     |notes                                                                         |month  |
+-----------------+------------+-------------+------------------------------------------------------------------------------+-------+
|1.0 (Launch)     |2020-12-10  |all          |Release; massive Bug-/Performance-Kritik, PS4/XB1-Version aus dem PSN entfernt|2020-12|
|1.1              |2021-01-22  |all          |Erster groesserer Stabilitaets-Patch                                          |2021-01|
|1.2              |2021-03-29  |all          |Grosser Bugfix-Patch (Fahrzeug-, Polizei-, KI-Fixes)                          |2021-03|
|1.3              |2021-08-17  |all          |Erstes Content-Update inkl. Fixes                                             |2021-08|
|1.5              |2022-02-15  |PS5/XSX + all|Next-Gen-Release

### Record Run Info
Which mode / which window the `results/` snapshot is based on (reproducibility).

In [8]:
from datetime import datetime, timezone

Path(RESULTS, "RUN_INFO.txt").write_text(
    f"DATA_MODE = {DATA_MODE}\n"
    f"window    = {WINDOW_START} .. {WINDOW_END}\n"
    f"subreddits= {SUBREDDITS}\n"
    f"seed      = {SEED}\n"
    f"generated = {datetime.now(timezone.utc).isoformat()}\n"
)
print((Path(RESULTS) / "RUN_INFO.txt").read_text())

DATA_MODE = full
window    = 2020-12-01 .. 2023-12-31
subreddits= ['cyberpunkgame', 'LowSodiumCyberpunk', 'Games', 'pcgaming']
seed      = 42
generated = 2026-06-11T00:44:56.649973+00:00



## 2 · Cleaning & Preprocessing  *(Phase 2 — TODO)*

- Normalize text (lowercasing, remove URLs/Markdown, whitespace)
- Language filter (English only — VADER is trained on English)
- Deduplication

## 3 · VADER Sentiment UDF  *(Phase 3 — TODO)*

VADER as Spark UDF on all comments/reviews; keep the `compound` score.

## 4 · Monthly/Source Aggregation  *(Phase 3 — TODO)*

Aggregate sentiment per `month` x `source` — **only here** does it become small enough for `.toPandas()`.

Writes: `results/sentiment_timeseries.csv` (`source, month, mean_sentiment, n`).

## 5 · Analysis & Visualization  *(Phase 3 — TODO)*

Each output -> exactly one file under `results/` (see report mapping in README):

- Two sentiment time series + patch markers  -> `results/figures/`, `results/patch_overlay.csv`
- Review-bombing anomaly flag (volume anomaly, `ANOMALY_Z_THRESHOLD`)  -> `results/anomaly_flags.csv`
- Sentiment without flagged windows  -> `results/sentiment_excl_bombing.csv`
- Recovery metric (core thesis)  -> `results/recovery_metric.csv`
- Topic analysis (TF-IDF / LDA with `setSeed(SEED)`, `LDA_NUM_TOPICS`)  -> `results/topics.csv`